In [ ]:
import pandas as pd

In [ ]:
txs = pd.read_csv('toy_data.csv',sep=r'\s*,\s*',index_col=False)
txs.head(2)

In [ ]:
real_user_accounts = ['Checking', 'Chase_CC']
real_proxy_accounts = []

# Mine Data

In [ ]:
income_txs = txs[txs['to'].isin(real_user_accounts)]
income_txs.head(2)

In [ ]:
real_outflow_txs = txs[ (txs['from'].isin(real_user_accounts))]
real_outflow_txs.head(2)

In [ ]:
proxy_outflow_txs = txs[ (txs['from'].isin(real_proxy_accounts))]
proxy_outflow_txs.head(2)

In [ ]:
incomes = income_txs.groupby(['from', 'to']).agg({'amount':'sum'}).reset_index()
incomes.head(2)

In [ ]:
outflows = real_outflow_txs.groupby(['from', 'to']).agg({'amount': 'sum'}).reset_index()
outflows.head(2)

In [ ]:
proxy_outflows = proxy_outflow_txs.groupby(['from', 'to']).agg({'amount': 'sum'}).reset_index()
proxy_outflows 

# Building out VTX

In [ ]:
import sbs
# We now try to match
# the virtual transactions to the 
# real transactions
rinc = pd.read_csv('toy_data.csv',sep=r'\s*,\s*', index_col=False)
# we can do a constraint reduction
# rtx.head(2)

# Build Visualizer

In [ ]:
import visualizer as vv

## Add Layers

In [ ]:
rinc['txid'][1]

In [ ]:
vinc = pd.read_csv('virtual_income.csv')
vinc.head(5)

In [ ]:
# get which account the income was transferred through by txid 
tinc = pd.merge(rinc, vinc, on='txid', suffixes=['_r', '_v'])
# convert to_r to "via"
tinc['via'] = tinc['to_r']
tinc.drop(columns=['from_r', 'to_r', 'amount_r'], inplace=True)
# figure out why chase cc is on here as a via
tinc.head(5)

In [ ]:
tinc.groupby(['from_v','via','to_v']).agg({'amount_v': 'sum'})

In [ ]:
########################
# Add Layers          #
#######################
vis = vv.LayerVisualizer()
ri = vis.add_layer('Real Income')
ra = vis.add_layer('Real Accounts')
va = vis.add_layer('Virtual Accounts')
raa = vis.add_layer('Real Accounts')
ro =vis.add_layer('Real Outflow')

########################
# Add Accounts         #
########################

# Real Income
for frm in set(income_txs['from']):
    ri.add_node(frm)
# Real Accounts
for to in set(income_txs['to']):
    ra.add_node(to)
# Virtual Accounts
for node in sorted(['Tuition', 'Groceries', 'Rent', 'Lease', 'Insurance', 'Free Cash'],reverse=True): va.add_node(node)
# Real Accounts (Reflow from Virtual Accounts)
for frm in set(real_outflow_txs['from']):
    raa.add_node(frm)
# Real Outflow
for to in set(real_outflow_txs['to']):
    ro.add_node(to)

########################
# Add Connections     #
########################
# construct proxy accounts
# build connections between layers
# build connection between ri and ra
# real accounts => proxy | outflow
for index, row in outflows.iterrows():
    frm = row['from']
    to = row['to']
    amount = row['amount']
    nfrm = raa.get_node(frm)
    try:
        nto = raa.get_node(to)
    except:
        nto = ro.get_node(to)
    nfrm.connect_to(nto, amount)

import random
random.seed(8262026)
def gen_hex() :
    color= (random.random(),random.random(),random.random())
    return color
tinc_balance_transfer = tinc.groupby(['from_v','via','to_v']).agg({'amount_v': 'sum'})
tinc_balance_transfer
print(set(rinc['from']))
colors = dict([(ri, gen_hex()) for ri in set(rinc['from'])])
# todo aggregate it by our ri account
for tx in tinc_balance_transfer.iterrows():
    cash_path, amount_series = tx
    amount = amount_series['amount_v']
    frm, via, to = cash_path
    # print(f'from {frm} ${amount} via {via} to {to}')

    color = colors[frm]
    ri.get_node(frm).connect_to(ra.get_node(via), color=color)
    ra.get_node(via).connect_to(va.get_node(to), color=color)

vis.show()

## Add Connections